# r/UofT labelling studio

MIE 1626 — Wednesday studio session.

**Goal.** Download a sample of posts from r/UofT, label each one as positive or negative about UofT, and cluster the positives and negatives separately.

**Two ways to do every step in this notebook.** The first is to write the code yourself. The second — and the *default* in this course — is to write a short Markdown spec and hand it to a coding agent (Claude Code, Codex, the VS Code agent, …). Both are shown side by side. Most of the time the agent route is faster and the quality is similar. The only catch is that you have to *read what it produced* and verify it, because the agent does silently helpful things sometimes (stop-word removal, label collapsing, …).

**Scale.** What's in this notebook is fine for a few hundred to a few thousand posts. For tens of thousands, you'd want batching, retry logic, on-disk caching, and a held-out hand-labelled eval set. We flag the spots where that would matter.

## Setup

In [ ]:
# pip install requests pandas openpyxl gensim scikit-learn matplotlib
# plus drop the notopenai folder (https://www.cs.toronto.edu/~guerzhoy/190/notopenai.zip)
# into the working directory or a parent of sys.path.

import json, time, random, re
from pathlib import Path

import requests
import pandas as pd
from notopenai import NotOpenAI

# Course key for the studio. Replace with your own once you have one.
NOTOPENAI_KEY = "XqlzORlA4NVDHlQtBugq"
client = NotOpenAI(api_key=NOTOPENAI_KEY)

DATA = Path("data")
DATA.mkdir(exist_ok=True)

## Step 1 — Download r/UofT posts via Arctic Shift

### The agentic way (default)

Write a Markdown spec like the one below into a new file called `spec.md` in your project folder, then say to Claude Code / Codex / the VS Code agent:

> Read `spec.md` and implement it. Save the data as `data/uoft_posts.json`. Don't filter anything yet — I want to eyeball the raw posts.

```
# r/UofT scrape spec

- Use the Arctic Shift API (https://arctic-shift.photon-reddit.com).
- Target subreddit: uoft.
- Fetch ~500 posts. Use pagination if needed.
- Keep at minimum: id, created_utc, title, selftext, score, num_comments.
- Write the result to data/uoft_posts.json as a list of dicts.
- Be polite: sleep ~1s between paginated requests.
```

### The DIY way (so you know what the agent will do)

In [ ]:
ARCTIC_URL = "https://arctic-shift.photon-reddit.com/api/posts/search"

def fetch_uoft(n_target=500, page=100, sleep_s=1.0):
    posts, before = [], None
    while len(posts) < n_target:
        params = {"subreddit": "uoft", "limit": page, "sort": "desc"}
        if before is not None:
            params["before"] = before
        r = requests.get(ARCTIC_URL, params=params, timeout=30)
        r.raise_for_status()
        batch = r.json().get("data", [])
        if not batch:
            break
        posts.extend(batch)
        before = batch[-1]["created_utc"]
        time.sleep(sleep_s)
    return posts[:n_target]

raw = fetch_uoft(n_target=500)
(DATA / "uoft_posts.json").write_text(json.dumps(raw, indent=2))
len(raw)

### Eyeball check

Before doing anything else, look at a handful of posts. If they don't look like r/UofT (e.g., wrong subreddit, all empty `selftext`), the rest of the pipeline is garbage.

In [ ]:
for p in random.sample(raw, 5):
    title = p.get("title", "")
    body = (p.get("selftext") or "").strip().replace("\n", " ")
    print("-", title)
    print(" ", body[:200], "..." if len(body) > 200 else "")
    print()

## Step 2 — Classify each post as positive or negative about UofT

We send each post's title + body to `gpt-3.5-turbo` via `notopenai` and ask for one of `positive` / `negative` / `neither`. The `neither` bucket matters: a lot of r/UofT posts are housing questions, lost items, course-suggestion threads, etc., that aren't really sentiment about UofT.

### Prompt design notes

- Constrain the output ("answer with one word"). The model will sometimes ramble; this keeps the parser simple.
- Give a tiny rubric. "Positive" and "negative" are subjective; pin them down with one-line definitions.
- Show the model the title *and* the body, but truncate the body — gpt-3.5 has limits and most r/UofT posts are short anyway.

### Rate limiting

`notopenai` rate-limits at roughly **one request every 2 seconds**. The helper below sleeps + retries on `Too many requests`, and the batch loop sleeps ~2.2s between calls. That makes 500 posts take ~18 minutes — plan accordingly. (Don't print `out.choices[0]` directly to debug — `notopenai`'s `__str__` chokes on non-JSON content. Use `out.choices[0].message.content`.)

In [ ]:
SYSTEM = (
    "You classify Reddit posts about the University of Toronto (UofT). "
    "Answer with exactly one word: 'positive', 'negative', or 'neither'. "
    "Use 'positive' if the post expresses satisfaction, praise, or a good experience with UofT. "
    "Use 'negative' if it expresses frustration, complaint, or a bad experience with UofT. "
    "Use 'neither' for neutral questions (e.g., 'where do I drop off this form'), "
    "lost-and-found posts, course recommendations, etc."
)

def call_with_retry(messages, max_retries=4, base_sleep=2.5):
    for attempt in range(max_retries):
        try:
            out = client.chat.completions.create(messages=messages, model="gpt-3.5-turbo")
            return out.choices[0].message.content
        except Exception as e:
            if "Too many requests" in str(e) and attempt < max_retries - 1:
                time.sleep(base_sleep * (attempt + 1))
                continue
            raise

def classify(title, body, max_body=600):
    user = f"Title: {title}\n\nBody: {(body or '')[:max_body]}"
    text = call_with_retry([
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": user},
    ]).strip().lower()
    # Defensive parsing: model sometimes wraps the word in quotes or punctuation.
    m = re.search(r"positive|negative|neither", text)
    return m.group(0) if m else "neither"

# Smoke test on one post.
sample = raw[0]
classify(sample.get("title", ""), sample.get("selftext", ""))

In [ ]:
# Batch over the whole download. ~2.2s/post (rate limit) => 500 posts ~ 18 min.
# For tens of thousands, you'd want async or a real batch API.
labels = []
for i, p in enumerate(raw):
    try:
        lab = classify(p.get("title", ""), p.get("selftext", ""))
    except Exception as e:
        print(f"[{i}] error: {e}")
        lab = "neither"
    labels.append(lab)
    time.sleep(2.2)
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(raw)}")

labelled = [dict(p, label=l) for p, l in zip(raw, labels)]
(DATA / "uoft_labelled.json").write_text(json.dumps(labelled, indent=2))

from collections import Counter
Counter(labels)

## Step 3 — Hand-verify a sample of labels in Excel

**You do not get to skip this.** This is the difference between a defensible analysis and a pile of noise.

The workflow:

1. Run the cell below to dump 50 random labelled posts to `data/hand_verify.xlsx`. The file has columns `id`, `title`, `body`, `model_label`, `your_label` (blank), `notes` (blank).
2. **Open `hand_verify.xlsx` in Excel.** Fill in `your_label` for each row — `positive`, `negative`, or `neither`. Use `notes` to flag anything ambiguous or to argue with the rubric.
3. Save the file (keep the same name, same sheet).
4. Run the read-back cell to compute model–you agreement.

If you disagree on more than ~15% of rows, the labels are not trustworthy. Sharpen the rubric in the Step 2 system prompt and re-run.

In [ ]:
random.seed(0)
sample = random.sample(labelled, 50)

verify_df = pd.DataFrame([
    {
        "id":          p.get("id"),
        "title":       p.get("title", ""),
        "body":        (p.get("selftext") or "").replace("\n", " ")[:600],
        "model_label": p["label"],
        "your_label":  "",
        "notes":       "",
    }
    for p in sample
])

verify_path = DATA / "hand_verify.xlsx"
verify_df.to_excel(verify_path, index=False)
print(f"Wrote {len(verify_df)} rows to {verify_path}. Open it in Excel and fill in `your_label`.")

### Read back and score

Run this after you've filled in `your_label` in Excel and saved.

In [ ]:
verified = pd.read_excel(DATA / "hand_verify.xlsx")
# Drop both NaN and empty-string rows.
mask = verified["your_label"].notna() & (verified["your_label"].astype(str).str.strip() != "")
done = verified[mask].copy()
done["your_label"] = done["your_label"].str.strip().str.lower()

agreement = (done["model_label"] == done["your_label"]).mean()
print(f"Hand-labelled {len(done)} / {len(verified)} rows.")
print(f"Agreement with the model: {agreement:.1%}")

# Where do we disagree? Useful for editing the system prompt.
disagree = done[done["model_label"] != done["your_label"]]
disagree[["title", "model_label", "your_label", "notes"]]

## Step 4 — Extract a "main keyword" per post

We want a single keyword per post to use as the unit of clustering. Asking the LLM for one keyword is the cheapest way to do this and works surprisingly well.

(You could do this in one combined call with Step 2 — "give me the label and the keyword in JSON" — and save half the LLM cost. We separate them here so the lecture stays readable.)

In [ ]:
KEYWORD_SYSTEM = (
    "Read a Reddit post and reply with one short lowercase noun phrase "
    "that best captures what the post is about. Examples: 'residence', "
    "'tuition', 'cs courses', 'mental health', 'tcard', 'orientation'. "
    "Reply with just the phrase. No quotes, no explanation."
)

def keyword(title, body, max_body=600):
    user = f"Title: {title}\n\nBody: {(body or '')[:max_body]}"
    text = call_with_retry([
        {"role": "system", "content": KEYWORD_SYSTEM},
        {"role": "user",   "content": user},
    ])
    return text.strip().lower().strip(".\"'")

for p in labelled:
    if p["label"] in ("positive", "negative"):
        try:
            p["keyword"] = keyword(p.get("title", ""), p.get("selftext", ""))
        except Exception:
            p["keyword"] = None
        time.sleep(2.2)

(DATA / "uoft_labelled_kw.json").write_text(json.dumps(labelled, indent=2))

Counter(p.get("keyword") for p in labelled if p["label"] == "negative").most_common(15)

## Step 5 — Cluster keywords with k-means on word vectors

This is the rough, lecture-time version of "what are the themes among complaints / praises." For each labelled post we have one keyword string; we look it up in a pretrained word-vector table; we run k-means.

**Caveats** (worth saying out loud in class):

- Multi-word keywords are averaged across their tokens. That's a crude bag-of-words embedding.
- A pretrained corpus (Wikipedia, common crawl) has no special knowledge of UofT. 'TCard' has no vector at all.
- We pick `k` by hand and inspect — there is no scoring step here. Doing this properly would use silhouette / elbow, and ultimately, *another round of LLM labelling on the cluster centroids.*

Treat this section as a sketch. The full version is left as a project-scale exercise.

In [ ]:
import gensim.downloader as gd
import numpy as np
from sklearn.cluster import KMeans

wv = gd.load("glove-wiki-gigaword-50")  # ~66MB on first run; cached after that

def embed(phrase):
    toks = re.findall(r"[a-z]+", phrase.lower())
    vecs = [wv[t] for t in toks if t in wv]
    if not vecs:
        return None
    return np.mean(vecs, axis=0)

def cluster_one_side(posts, k=5):
    items = [(p["keyword"], embed(p["keyword"])) for p in posts if p.get("keyword")]
    items = [(kw, v) for kw, v in items if v is not None]
    if not items:
        return []
    keywords = [kw for kw, _ in items]
    X = np.stack([v for _, v in items])
    k_eff = min(k, len(items))  # KMeans errors if k > n
    km = KMeans(n_clusters=k_eff, n_init=10, random_state=0).fit(X)
    return list(zip(keywords, km.labels_))

neg_posts = [p for p in labelled if p["label"] == "negative"]
pos_posts = [p for p in labelled if p["label"] == "positive"]

neg_clusters = cluster_one_side(neg_posts, k=5)
pos_clusters = cluster_one_side(pos_posts, k=5)

In [ ]:
def show(name, clusters):
    print(f"=== {name} ===")
    by_cluster = {}
    for kw, c in clusters:
        by_cluster.setdefault(c, []).append(kw)
    for c, kws in sorted(by_cluster.items()):
        # show a handful of representative keywords per cluster
        sample = list(dict.fromkeys(kws))[:10]
        print(f"  cluster {c} (n={len(kws)}): {', '.join(sample)}")
    print()

show("NEGATIVE clusters", neg_clusters)
show("POSITIVE clusters", pos_clusters)

## Closing — what would the production version look like?

Everything above is a one-afternoon studio. If you wanted to actually publish a claim like "the top three things UofT students complain about on Reddit are X, Y, Z," you would need:

1. **A held-out hand-labelled eval set.** A few hundred posts, labelled by you (or by classmates with an agreement check). Reported model accuracy on this set is what makes the rest of the pipeline defensible.
2. **A proper batched API call path.** `gpt-3.5-turbo` has both a Batch API and an async client. Running 50,000 posts one-at-a-time at 1 RPS takes 14 hours and burns money on overhead. Batching gets you 10x.
3. **Robust retry + on-disk caching.** Network blips will happen; you don't want to re-pay for a label that already exists.
4. **Stratification by time and by post type.** Sentiment on r/UofT in September (orientation) is not the same as in April (exams).
5. **A better clustering step.** Either embed the *whole post* with a sentence encoder and cluster those, or have the LLM cluster the keywords itself ("group these 200 keywords into 5–8 themes"). The word-vector route here is the cheap version that fits in a studio.

For Mini-Project / Project work in this course: items 1 and 3 are the ones to do. Items 2, 4, 5 are upgrades you'd do only if the project depends on them.